In [ ]:
!mkdir -p project/agents project/tools project/memory project/core

!touch project/__init__.py project/agents/__init__.py project/tools/__init__.py project/memory/__init__.py project/core/__init__.py


In [ ]:
%%writefile project/agents/planner.py
class Planner:
    def plan(self, user_input):
        return {
            "task": user_input,
            "steps": [
                "Analyze emergency request",
                "Execute required tools",
                "Validate final response"
            ]
        }


In [ ]:
%%writefile project/agents/worker.py
from project.tools.tools import emergency_tool, summarize_text

class Worker:
    def execute(self, plan):
        info = emergency_tool(plan["task"])
        summary = summarize_text(info)
        return summary


In [ ]:
%%writefile project/agents/evaluator.py
class Evaluator:
    def evaluate(self, response):
        return {
            "approved": True,
            "response": response
        }


In [ ]:
%%writefile project/tools/tools.py
def emergency_tool(query):
    return f"Emergency resource lookup completed for: {query}"

def summarize_text(text):
    return f"Summary: {text}"


In [ ]:
%%writefile project/memory/session_memory.py
class SessionMemory:
    def __init__(self):
        self.history = []

    def add(self, data):
        self.history.append(data)

    def get(self):
        return self.history


In [ ]:
%%writefile project/core/context_engineering.py
def build_context(user_input):
    return {
        "user_input": user_input,
        "priority": "medium"
    }


In [ ]:
%%writefile project/core/observability.py
def log_event(message):
    print("[LOG]", message)


In [ ]:
%%writefile project/core/a2a_protocol.py
def send_message(sender, receiver, message):
    return {
        "sender": sender,
        "receiver": receiver,
        "message": message
    }


In [ ]:
%%writefile project/main_agent.py
from project.agents.planner import Planner
from project.agents.worker import Worker
from project.agents.evaluator import Evaluator
from project.memory.session_memory import SessionMemory
from project.core.context_engineering import build_context
from project.core.observability import log_event

class MainAgent:
    def __init__(self):
        self.planner = Planner()
        self.worker = Worker()
        self.evaluator = Evaluator()
        self.memory = SessionMemory()

    def handle_message(self, user_input):
        context = build_context(user_input)
        self.memory.add(context)

        log_event("Planning started")

        plan = self.planner.plan(user_input)
        result = self.worker.execute(plan)
        checked = self.evaluator.evaluate(result)

        return checked

def run_agent(user_input: str):
    agent = MainAgent()
    result = agent.handle_message(user_input)
    return result["response"]


In [ ]:
%%writefile project/app.py
from project.main_agent import run_agent

def app():
    return run_agent("Emergency assistance request")

if __name__ == "__main__":
    print(app())


In [ ]:
%%writefile project/run_demo.py
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname(__file__), "..")))
from project.main_agent import run_agent

if __name__ == "__main__":
    print(run_agent("Hello! This is a demo."))


In [ ]:
%%writefile project/requirements.txt
streamlit


In [ ]:
from project.main_agent import run_agent
print(run_agent("Hello!"))


In [ ]:
!zip -r project.zip project
